In [ ]:
import pandas as pd
import numpy as np
import altair as alt

# Using ecostyles package (installed in environment)
from ecostyles import EcoStyles
styles = EcoStyles()
styles.register_and_enable_theme(theme_name='article')

In [ ]:
oecd2024 = pd.read_csv("oecdtaxdata2024_categories.csv")
oecdpanel = pd.read_csv("oecdtaxdata_panel.csv")

In [ ]:
columnskeep = ['REF_AREA','Reference area','Institutional sector','OBS_VALUE']
oecd2024 = oecd2024[columnskeep]
oecd2024

In [ ]:
#computing the percentage of tax revenue collected by central government

# pivot: one row per country, sectors become columns
df_wide = oecd2024.pivot_table(
    index=['REF_AREA', 'Reference area'],
    columns='Institutional sector',
    values='OBS_VALUE',
    aggfunc='first'  # in case of duplicates, adjust if you have multiple years/periods
).reset_index()

# clean up column names (pivot_table can leave a MultiIndex/name on columns)
df_wide.columns.name = None

# replace missing government-sector values with 0 so ratios are computed cleanly
df_wide[['Central government', 'State government', 'Local government', 'European institutions and bodies (EU accounts)']] = (
    df_wide[['Central government', 'State government', 'Local government','European institutions and bodies (EU accounts)']]
    .fillna(0)
)

# compute ratio; zero values now mean missing sectors are treated as 0
df_wide['central_to_overall_ratio'] = (
    df_wide['Central government'] / (df_wide['General government'])
)

df_wide = df_wide[df_wide['Reference area'] != 'Japan'] # removing japan as there is no 2024 SS data

df_wide

In [ ]:
bars = alt.Chart(df_wide).mark_bar().encode(
    x=alt.X(
        'Reference area:N',
        sort='-y',
        axis=alt.Axis(labelAngle=-45)
    ),
    y=alt.Y(
        'central_to_overall_ratio:Q',
        axis=alt.Axis(format='%', title=None)
    ),
    color=alt.condition(
        alt.datum.REF_AREA == 'GBR',
        alt.value('#d90000'),
        alt.value('#179fdb')
    ),
    tooltip=[alt.Tooltip('Reference area', title='Country'), alt.Tooltip('central_to_overall_ratio:Q', title='Central Government Revenue Share', format='.1%')]
).properties(
    width=800,
    height=350,
    title='Central government share of total government tax revenue'
)

source_note = alt.Chart(pd.DataFrame({'note': ['Data from OECD. Japan is omitted due to missing 2024 social security revenue data.']})).mark_text(
    align='left',
    baseline='middle',
    color='gray'
).encode(
    x=alt.value(0),
    text='note:N'
).properties(
    width=800,
    height=25
)

chart = alt.vconcat(bars, source_note)

chart

In [ ]:
styles.save(chart, 'charts', 'oecdtaxcbarchart', width=600, height=200)

In [ ]:
df_wide_stacked = df_wide.assign(
    central_share=df_wide["central_to_overall_ratio"],
    other_share=1 - df_wide["central_to_overall_ratio"]
)

df_stacked = df_wide_stacked.melt(
    id_vars=["REF_AREA", "Reference area", "central_to_overall_ratio"],
    value_vars=["central_share", "other_share"],
    var_name="share_type",
    value_name="share"
)

df_stacked["share_type"] = df_stacked["share_type"].replace({
    "central_share": "Central government",
    "other_share": "State, local, and other"
})

bars2 = alt.Chart(df_stacked).mark_bar().encode(
    x=alt.X(
        "Reference area:N",
        sort=alt.SortField("central_to_overall_ratio", order="descending"),
        axis=alt.Axis(labelAngle=-45)
    ),
    y=alt.Y(
        "share:Q",
        axis=alt.Axis(format="%", title=None),
        stack="normalize"
    ),
    color=alt.Color(
        "share_type:N",
        scale=alt.Scale(
            domain=["State, local, and other", "Central government"],
            range=["#179fdb", "#143c54"]
        ),
        title=None
    ),
    opacity=alt.condition(
        alt.datum.REF_AREA == "GBR",
        alt.value(1),
        alt.value(0.8)
    ),
    tooltip=[
        alt.Tooltip("Reference area", title="Country"),
        alt.Tooltip("share_type:N", title="Component"),
        alt.Tooltip("share:Q", title="Share", format=".1%")
    ]
).properties(
    width=800,
    height=350,
    title="Central government share of total government tax revenue"
)

chart2 = alt.vconcat(bars2, source_note)
chart2

In [ ]:
styles.save(chart2, 'charts', 'oecdtaxcbarchart_cat', width=600, height=200)